# Pokémon TCG single-expert behavior cloning

This private worker processes exactly one frozen expert policy: seed exact Episode IDs from attached daily replay Datasets, download only missing frozen Episodes through the API, build and audit the universal full-action dataset, train the shared BC architecture with validation-only LR rescue selection, evaluate the selected checkpoint on the frozen test split, and export a standard `main.py` + `deck.csv` + `cg/` package.

Package validation and offline imitation metrics are not strength claims. Run the downloaded candidate against the official engine locally before adding it to `evaluation/opponents`.

In [ ]:
# Change only JOB_ORDER between private Kernel versions. Orders 1 and 2 are
# intentionally absent because they were already trained locally.
JOB_ORDER = 3
WORKER_MODE = "full"  # "prepare" on CPU or "train" from the prepare Kernel output.
CORPUS_MODE = "exact_submission"  # use "daily_team_winners" to mirror the reference Yushin corpus.
PREBUILT_KERNEL_SOURCE = ""
RETENTION = "dataset"  # "dataset" keeps processed JSONL; "package" keeps only model/package evidence.
OUTPUT_ROOT = "/kaggle/working/ptcg_bc_outputs"
EMBEDDED_INPUT_ARCHIVE_B64 = ""
EMBEDDED_INPUT_ARCHIVE_SHA256 = ""


In [ ]:
%pip install --quiet "kaggle==2.2.3" "kagglesdk>=0.1.33,<1.0" "numpy<2" "tensorboard>=2.14"

In [ ]:
import base64
import hashlib
import io
import os
import shutil
import tarfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/working/ptcg_bc_input")
if EMBEDDED_INPUT_ARCHIVE_B64:
    archive_bytes = base64.b64decode(EMBEDDED_INPUT_ARCHIVE_B64, validate=True)
    actual_archive_hash = hashlib.sha256(archive_bytes).hexdigest()
    if actual_archive_hash != EMBEDDED_INPUT_ARCHIVE_SHA256:
        raise RuntimeError("Embedded BC input archive hash mismatch")
    INPUT_ROOT.mkdir(parents=True, exist_ok=False)
    with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as handle:
        for member in handle.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Archive links are not allowed: {member.name}")
            if not member.isfile() and not member.isdir():
                raise RuntimeError(f"Unsupported archive member: {member.name}")
            target = (INPUT_ROOT / member.name).resolve()
            if INPUT_ROOT.resolve() not in (target, *target.parents):
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        handle.extractall(INPUT_ROOT)
else:
    private_input_markers = [
        *Path("/kaggle/input").glob("datasets/tommycyd/pokemon-tcg-bc-cloud-input/ptcg_kaggle_bc_input/ptcg_kaggle_bc_input.json"),
        *Path("/kaggle/input").glob("datasets/tommycyd/pokemon-tcg-bc-cloud-input/versions/*/ptcg_kaggle_bc_input/ptcg_kaggle_bc_input.json"),
    ]
    private_input_markers = [path for path in private_input_markers if path.is_file()]
    if len(private_input_markers) != 1:
        raise RuntimeError(f"Expected one extracted private BC input, found {private_input_markers}")
    shutil.copytree(private_input_markers[0].parent, INPUT_ROOT)
if not (INPUT_ROOT / "ptcg_kaggle_bc_input.json").is_file():
    raise RuntimeError("Extracted BC input marker is missing")
official_cg_candidates = [
    Path("/kaggle/input/competitions/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg"),
    Path("/kaggle/input/pokemon-tcg-ai-battle/sample_submission/sample_submission/cg"),
]
official_cg = [path for path in official_cg_candidates if path.is_dir()]
if len(official_cg) != 1:
    discovered = list(Path("/kaggle/input").glob("**/sample_submission/sample_submission/cg"))
    official_cg = [path for path in discovered if path.is_dir()]
if len(official_cg) != 1:
    raise RuntimeError(f"Expected one official competition cg runtime, found {official_cg}")
shutil.copytree(official_cg[0], INPUT_ROOT / "runtime/cg")
REPO_ROOT = INPUT_ROOT / "repo"
official_card_candidates = [
    Path("/kaggle/input/competitions/pokemon-tcg-ai-battle/EN_Card_Data.csv"),
    Path("/kaggle/input/pokemon-tcg-ai-battle/EN_Card_Data.csv"),
]
official_cards = [path for path in official_card_candidates if path.is_file()]
if len(official_cards) != 1:
    raise RuntimeError(f"Expected one official EN_Card_Data.csv, found {official_cards}")
card_destination = REPO_ROOT / "data/official/EN_Card_Data.csv"
card_destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(official_cards[0], card_destination)
os.environ["PYTHONPATH"] = str(REPO_ROOT)
mounted_replay_datasets = []
if WORKER_MODE in {"full", "prepare"} and CORPUS_MODE == "daily_team_winners":
    mounted_replay_datasets = [Path("/kaggle/input")]
elif WORKER_MODE in {"full", "prepare"}:
    mounted_replay_datasets = sorted([
        *Path("/kaggle/input").glob("pokemon-tcg-ai-battle-episodes-*"),
        *Path("/kaggle/input").glob("datasets/kaggle/pokemon-tcg-ai-battle-episodes-*"),
        *Path("/kaggle/input").glob("datasets/kaggle/pokemon-tcg-ai-battle-episodes-*/versions/*"),
    ])
    mounted_replay_datasets = [path for path in mounted_replay_datasets if path.is_dir() and next(path.glob("[0-9]*.json"), None) is not None]
    if not mounted_replay_datasets:
        raise RuntimeError("No daily Episode datasets are attached")
PREBUILT_ROOT = None
if WORKER_MODE == "train":
    prepare_results = sorted(Path("/kaggle/input").rglob("PREPARE_RESULT.json"))
    prepare_results = [path for path in prepare_results if path.is_file()]
    if len(prepare_results) != 1:
        raise RuntimeError(f"Expected one mounted PREPARE_RESULT.json, found {prepare_results}")
    PREBUILT_ROOT = prepare_results[0].parent
print("Input bundle:", INPUT_ROOT)
print("Mounted daily Episode datasets:", [path.name for path in mounted_replay_datasets])
print("Prebuilt prepare root:", PREBUILT_ROOT)
print("Worker mode:", WORKER_MODE)
print("Job order:", JOB_ORDER)

In [ ]:
# Prefer the current access-token secret. Legacy username/key secrets remain
# supported for accounts that have not migrated yet. Secret values are never printed.
if WORKER_MODE in {"full", "prepare"}:
 try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            value = secrets.get_secret(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value
 except ImportError:
    pass
print("Kaggle API credentials are required only for Dataset misses in prepare mode.")

In [ ]:
import subprocess
import sys

command = [
    sys.executable, "-m", "train.kaggle_bc_top20.worker",
    "--input-root", str(INPUT_ROOT),
    "--output-root", OUTPUT_ROOT,
    "--job-order", str(JOB_ORDER),
    "--device", "cpu" if WORKER_MODE == "prepare" else "cuda",
    "--mode", WORKER_MODE,
    "--corpus-mode", CORPUS_MODE,
    "--retention", RETENTION,
]
if WORKER_MODE in {"full", "prepare"}:
    command.extend(["--mounted-replay-root", "/kaggle/input"])
if WORKER_MODE == "train":
    command.extend(["--prebuilt-root", str(PREBUILT_ROOT)])
process = subprocess.Popen(
    command,
    cwd=REPO_ROOT,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code:
    raise RuntimeError(f"BC worker failed with exit code {return_code}")

In [ ]:
import json

result_name = "PREPARE_RESULT.json" if WORKER_MODE == "prepare" else "RESULT.json"
results = list(Path(OUTPUT_ROOT).rglob(result_name))
if len(results) != 1:
    raise RuntimeError(f"Expected one RESULT.json, found {results}")
result = json.loads(results[0].read_text())
summary = {
    "status": result["status"],
    "package_name": result["job"]["package_name"],
    "dataset_sha256": result["dataset"]["sha256"],
    "official_game_evaluation_performed": result["official_game_evaluation_performed"],
}
if result["status"] == "candidate_ready":
    summary.update({
        "offline_gate_passed": result["offline_gate_passed"],
        "candidate_archive": result["candidate"]["archive"],
        "candidate_archive_sha256": result["candidate"]["archive_sha256"],
    })
print(json.dumps(summary, indent=2))